# 03 — Full measurement pipeline on an AnnData object

The `measure_operator` function is the top-level entry point that accepts an AnnData-like object
directly, handles guide→target routing, builds guide responses via the efficiency estimator regime,
and returns a `MeasuredOperator` bundled with an `AnchorReport`.

This tutorial builds a synthetic AnnData-like object with the schema `measure_operator` expects
(same as a real Perturb-seq h5ad), runs the pipeline, and walks through every field of the returned
report.

In [ ]:
import numpy as np
import pandas as pd
from types import SimpleNamespace
import anchorop as ao
rng = np.random.default_rng(0)

## 1. Simulate a Perturb-seq-like dataset

We construct a small AnnData-like object with:
- `X`: cells × genes (log-normalized count-like values)
- `obs["guide"]`: sgRNA identifier per cell (`"non-targeting"` for controls)
- `obs["target_gene"]`: target gene symbol (empty string for controls)
- `var_names`: gene symbols

Under the hood we simulate: cells drawn from a known additive-input linear model with per-guide
knockdown κ, plus per-cell Gaussian noise.

In [ ]:
def simulate_perturbseq(n_ctrl=500, n_targets=10, sgrnas_per_target=3, cells_per_sgrna=40, d=10, n_genes=200, rng=rng):
    # Ground-truth Jacobian in a d-dim program space
    J = rng.normal(0.0, 1.0, size=(d, d)) - 3.0 * np.eye(d)
    # Gene loadings for the program basis: each gene projects onto d programs
    W = rng.normal(size=(n_genes, d)) / np.sqrt(d)
    # Cell-level noise scale
    sigma_cell = 0.5

    # Build cell metadata
    cell_rows = []; X_rows = []
    # Controls
    for _ in range(n_ctrl):
        cell_rows.append({"guide": "non-targeting", "target_gene": ""})
        X_rows.append(sigma_cell * rng.normal(size=n_genes))
    # Perturbed cells
    target_names = [f"GENE_{i:03d}" for i in range(n_targets)]
    for t_idx, tgt in enumerate(target_names):
        for sg in range(sgrnas_per_target):
            kappa = 0.3 + 0.6 * rng.uniform()
            # δ_t: perturbation direction in program space is Wᵀ[e_{target_gene}]
            e_t = np.zeros(n_genes); e_t[t_idx] = 1.0
            u = -kappa * W.T @ e_t
            # Steady-state response in program coordinates
            dz = -np.linalg.solve(J, u)
            # Gene-level shift
            gene_shift = W @ dz
            for _ in range(cells_per_sgrna):
                cell_rows.append({"guide": f"sg_{tgt}_{sg:02d}", "target_gene": tgt})
                X_rows.append(gene_shift + sigma_cell * rng.normal(size=n_genes))

    X = np.array(X_rows)
    obs = pd.DataFrame(cell_rows)
    var_names = np.array([f"GENE_{i:03d}" if i < n_targets else f"BG_{i:03d}" for i in range(n_genes)])
    return SimpleNamespace(X=X, obs=obs, var_names=var_names), J, W

adata, J_true, W_true = simulate_perturbseq()
print(f"adata: {adata.X.shape} cells × genes")
print(f"guides: {adata.obs['guide'].nunique()} (including 1 non-targeting)")
print(f"controls: {(adata.obs['guide'] == 'non-targeting').sum()} cells")

## 2. Provide a program basis

You can either fit one from controls (`ao.fit_programs`) or supply an external basis via
`ao.make_program_basis`. Here we build one from the true generative loadings for pedagogical clarity;
in practice you would fit from data (see tutorial 06).

In [ ]:
basis = ao.make_program_basis(
    W_true.astype(np.float32),        # gene × program loading matrix
    gene_names=adata.var_names,
    method="external",
    control_count=int((adata.obs['guide'] == 'non-targeting').sum()),
    normalize=False,
)
print(f"basis: d = {basis.d}, n_genes = {basis.n_genes}")

## 3. Run measure_operator

Every argument shown here has a sensible default; the ones we set are the most commonly-customized:

- `guide_key`, `target_key`, `control_label` — obs column names and control identifier
- `efficiency_estimator="auto"` — data-format-aware router (see tutorial 02)
- `rank_tol=1e-2` — preregistered identifiability guard
- `bootstrap=100` — guide-bootstrap for uncertainty (0 to disable)

In [ ]:
m = ao.measure_operator(
    adata, basis,
    guide_key="guide", target_key="target_gene", control_label="non-targeting",
    min_cells_per_guide=10,
    min_knockdown_efficiency=0.05,
    reg="tsvd", reg_param="path",
    rank_tol=1e-2,
    bootstrap=100, bootstrap_seed=0,
    efficiency_estimator="auto",
)

r = m.report
print(f"retained {r.n_guides_retained}/{r.n_guides_input} guides")
print(f"effective response rank: {r.effective_response_rank}/{r.d}")
print(f"full domain identified:  {r.full_domain_identified}")
print(f"condition number:        {r.condition_number:.2f}")
print(f"regularization method:   {r.regularization_method}")
print(f"selected parameter:      {r.selected_regularization}")

## 4. Read the AnchorReport

The report fields are documented in `anchorop.types.AnchorReport`. Key ones:

- **Identifiability**: `d`, `effective_response_rank`, `full_domain_identified`, `rank_tol`
- **Guide accounting**: `n_guides_input`, `n_guides_retained`, `retained_guides`, `dropped_guides` (dict of guide → drop reason)
- **Regularization**: `regularization_path` (all rank levels), `selected_regularization`, `singular_values`
- **Projectors**: `input_projector`, `response_projector` (both d×d, describe which subspaces are covered)
- **Uncertainty**: `bootstrap_covariance` (d²×d²), `bootstrap_actions` (n_bootstrap × d × d) — populated iff `bootstrap > 0`
- **Efficiencies**: `guide_efficiencies` (guide → κ used in the fit)
- **Provenance**: `notes` (which estimator ran, which auto-routing happened, etc.)

In [ ]:
# Drop-reason accounting
from collections import Counter
drop_reasons = Counter(r.dropped_guides.values())
if drop_reasons:
    print("Guide drop reasons:")
    for reason, count in drop_reasons.items():
        print(f"  {count:>4d}  {reason}")

# Provenance notes (auto-router records what it chose)
print()
print("report.notes:")
for note in r.notes:
    print(f"  - {note}")

## 5. Access the operator (safely)

If `full_domain_identified` is `True`, `.J` returns the full Jacobian. Otherwise it raises
`IdentifiabilityError` — this is the type-level guard against overclaiming.

In [ ]:
if r.full_domain_identified:
    J_fit = m.J
    print(f"J_fit shape: {J_fit.shape}")
    print(f"Frobenius error vs truth: {np.linalg.norm(J_fit - J_true) / np.linalg.norm(J_true):.4f}")
else:
    # On partial-rank measurements, use .identified_action (returns J·P_X)
    print("Partial identification; use m.identified_action (J·P_X) instead")
    print(f"identified_action shape: {m.identified_action.shape}")

## 6. Uncertainty from the bootstrap

`bootstrap_actions` is a stack of Jacobians (one per bootstrap replicate). Per-entry standard
error is the empirical std across the stack. Per-column norms give per-guide uncertainty.

In [ ]:
if r.bootstrap_actions is not None:
    boot = r.bootstrap_actions             # (n_bootstrap, d, d)
    A_hat = m.identified_action
    se = boot.std(axis=0)                  # per-entry SE
    print(f"bootstrap replicates: {boot.shape[0]}")
    print(f"||A_hat||_F           = {np.linalg.norm(A_hat):.3f}")
    print(f"per-entry SE:  mean = {se.mean():.4f}, max = {se.max():.4f}")
    # Relative SE (SE / |entry|) tells you how reproducibly individual entries are estimated
    rel_se = se / (np.abs(A_hat) + 1e-9)
    print(f"per-entry relative SE:  median = {np.median(rel_se):.2f}, 90th pct = {np.percentile(rel_se, 90):.2f}")

## Common pitfalls

- **Guide efficiency spikes to 1.0** on many targets → your data is count data with low-baseline
  targets. Check estimator routing (`report.notes`) and consider raising `min_control_detection_rate`.
- **`full_domain_identified` is False** → your guide library doesn't span the requested `d` dimensions
  in program space. Reduce `d`, or accept partial identification and work with `.identified_action`.
- **Very high condition number (>100)** → the pseudoinverse is amplifying noise on the smallest
  retained direction. Consider raising `rank_tol` (fewer directions, better conditioning) or
  increasing `d` (retain more real signal per direction).

## Next

- **04**: how to test whether the linear-response model is defensible on your data.
- **05**: comparing anchor-op's output against inferred operators from other methods.
- **06**: fitting a program basis from your own data (NMF / cNMF workflow).